# De MediaPipe vers SMPL — Conversion cinématique
## Tutoriel 2 / 4

MediaPipe donne des **positions** de landmarks (où sont les points).
SMPL a besoin de **rotations articulaires** (comment chaque os est orienté).

Ce notebook montre comment passer de l'un à l'autre par mapping cinématique direct.

```
Landmarks 3D (positions)  →  Vecteurs os  →  Rotations axis-angle  →  body_pose SMPL
```

> **Note** : Cette conversion est une *approximation*. Elle ne fait pas de fitting
> de forme (`betas = 0`) et n'utilise pas de modèle SMPL complet.
> GVHMR fait mieux en optimisant les paramètres sur chaque frame,
> mais cette approche est 100% locale, sans GPU, et pédagogiquement instructive.

In [ ]:
!pip install mediapipe opencv-python-headless matplotlib --quiet

import cv2
import mediapipe as mp
import numpy as np
import torch
import matplotlib.pyplot as plt
from scipy.spatial.transform import Rotation as sRot
from pathlib import Path

mp_pose = mp.solutions.pose
print("Prêt.")

## 1. Rappel — structure SMPL vs MediaPipe

SMPL modélise le corps avec **24 os** disposés en arbre depuis la racine (pelvis).
Chaque os est décrit par sa rotation locale par rapport à son parent.

MediaPipe donne **33 landmarks** dont 25 correspondent approximativement
aux joints SMPL. On va utiliser cette correspondance.

In [ ]:
# Correspondance MediaPipe → os SMPL
# Format : nom_smpl → (indice_proximal_mp, indice_distal_mp)
# L'orientation de l'os = direction du vecteur proximal→distal

MP = mp_pose.PoseLandmark

SMPL_BONES = {
    # Jambe gauche
    "L_Hip":   (MP.LEFT_HIP,   MP.LEFT_KNEE),
    "L_Knee":  (MP.LEFT_KNEE,  MP.LEFT_ANKLE),
    "L_Ankle": (MP.LEFT_ANKLE, MP.LEFT_FOOT_INDEX),
    # Jambe droite
    "R_Hip":   (MP.RIGHT_HIP,   MP.RIGHT_KNEE),
    "R_Knee":  (MP.RIGHT_KNEE,  MP.RIGHT_ANKLE),
    "R_Ankle": (MP.RIGHT_ANKLE, MP.RIGHT_FOOT_INDEX),
    # Torse
    "Spine":   (MP.LEFT_HIP,    MP.RIGHT_HIP),       # approximation
    "Chest":   (MP.LEFT_SHOULDER, MP.RIGHT_SHOULDER),
    # Bras gauche
    "L_Shoulder": (MP.LEFT_SHOULDER, MP.LEFT_ELBOW),
    "L_Elbow":    (MP.LEFT_ELBOW,    MP.LEFT_WRIST),
    # Bras droit
    "R_Shoulder": (MP.RIGHT_SHOULDER, MP.RIGHT_ELBOW),
    "R_Elbow":    (MP.RIGHT_ELBOW,    MP.RIGHT_WRIST),
    # Tête
    "Neck":  (MP.LEFT_SHOULDER, MP.NOSE),
}

# Pose de référence SMPL en T-pose (vecteurs os dans le repère local)
# direction +Y = vers le haut pour les membres inférieurs dans le repère SMPL
T_POSE_DIRS = {
    "L_Hip":      np.array([ 0, -1,  0]),   # cuisse vers le bas
    "L_Knee":     np.array([ 0, -1,  0]),
    "L_Ankle":    np.array([ 0, -1,  0]),
    "R_Hip":      np.array([ 0, -1,  0]),
    "R_Knee":     np.array([ 0, -1,  0]),
    "R_Ankle":    np.array([ 0, -1,  0]),
    "Spine":      np.array([ 1,  0,  0]),   # de gauche à droite
    "Chest":      np.array([ 1,  0,  0]),
    "L_Shoulder": np.array([-1,  0,  0]),   # bras gauche vers la gauche
    "L_Elbow":    np.array([-1,  0,  0]),
    "R_Shoulder": np.array([ 1,  0,  0]),   # bras droit vers la droite
    "R_Elbow":    np.array([ 1,  0,  0]),
    "Neck":       np.array([ 0,  1,  0]),   # tête vers le haut
}

print(f"{len(SMPL_BONES)} os mappés MediaPipe → SMPL")

## 2. Calcul des rotations

Pour chaque os, on calcule la rotation qui transforme
la direction de repos (T-pose) vers la direction observée.
C'est une rotation minimale dans l'espace 3D → représentée en axis-angle.

In [ ]:
def vec_to_vec_rotation(v_from, v_to):
    """
    Rotation minimale (axe-angle) qui oriente v_from vers v_to.
    Retourne un vecteur axis-angle (3,) — norme = angle en radians.
    """
    v_from = v_from / (np.linalg.norm(v_from) + 1e-9)
    v_to   = v_to   / (np.linalg.norm(v_to)   + 1e-9)

    dot = np.clip(np.dot(v_from, v_to), -1.0, 1.0)
    angle = np.arccos(dot)         # angle de rotation

    if angle < 1e-6:
        return np.zeros(3)         # déjà alignés

    axis = np.cross(v_from, v_to)  # axe de rotation
    axis_norm = np.linalg.norm(axis)

    if axis_norm < 1e-9:           # vecteurs opposés
        # Choisir un axe perpendiculaire arbitraire
        axis = np.array([1, 0, 0]) if abs(v_from[0]) < 0.9 else np.array([0, 1, 0])
    else:
        axis = axis / axis_norm

    return axis * angle            # axis-angle


def landmarks_to_smpl_pose(landmarks_frame):
    """
    Convertit les landmarks 3D d'une frame en body_pose SMPL approximatif.

    Args:
        landmarks_frame : (33, 4) array — x, y, z, visibility

    Returns:
        body_pose : (69,) array — 23 rotations axis-angle (sans la racine)
        global_orient : (3,) array — rotation racine
        transl : (3,) array — translation (position des hanches)
    """
    xyz = landmarks_frame[:, :3]   # (33, 3)

    # Rotation globale : orientation du bassin
    left_hip  = xyz[MP.LEFT_HIP.value]
    right_hip = xyz[MP.RIGHT_HIP.value]
    left_sh   = xyz[MP.LEFT_SHOULDER.value]

    hip_axis   = right_hip - left_hip              # axe X local (gauche→droite)
    spine_axis = (left_sh + (xyz[MP.RIGHT_SHOULDER.value])) / 2 - \
                 (left_hip + right_hip) / 2        # axe Y local (bas→haut)

    hip_axis   = hip_axis   / (np.linalg.norm(hip_axis)   + 1e-9)
    spine_axis = spine_axis / (np.linalg.norm(spine_axis) + 1e-9)
    z_axis     = np.cross(hip_axis, spine_axis)
    z_axis     = z_axis / (np.linalg.norm(z_axis) + 1e-9)
    # Orthogonalisation
    spine_axis = np.cross(z_axis, hip_axis)

    rot_mat = np.stack([hip_axis, spine_axis, z_axis], axis=1)  # 3×3
    try:
        global_orient = sRot.from_matrix(rot_mat).as_rotvec()   # (3,)
    except Exception:
        global_orient = np.zeros(3)

    # Translation : position du milieu des hanches
    transl = (left_hip + right_hip) / 2

    # Rotations des 13 os mappés → convertir en 23 rotations SMPL
    # Ordre SMPL body_pose : L_Hip, R_Hip, Spine, L_Knee, R_Knee, ...
    # On remplit les os non mappés avec des rotations nulles
    SMPL_JOINT_ORDER = [
        "L_Hip", "R_Hip", "Spine",
        "L_Knee", "R_Knee", "Chest",
        "L_Ankle", "R_Ankle", None,    # Neck (torse)
        None, None, "Neck",             # L_Toe, R_Toe, Cou
        None, "L_Shoulder", "R_Shoulder",
        None, "L_Elbow", "R_Elbow",
        None, "L_Wrist", "R_Wrist",
        None, None,
    ]

    body_pose = np.zeros(23 * 3)
    for j, bone_name in enumerate(SMPL_JOINT_ORDER):
        if bone_name is None or bone_name not in SMPL_BONES:
            continue

        prox_idx, dist_idx = SMPL_BONES[bone_name]
        prox = xyz[prox_idx.value]
        dist = xyz[dist_idx.value]
        bone_dir_world = dist - prox

        # Projeter dans le repère local du pelvis
        bone_dir_local = rot_mat.T @ bone_dir_world

        # Rotation minimale depuis la T-pose
        rotvec = vec_to_vec_rotation(T_POSE_DIRS[bone_name], bone_dir_local)
        body_pose[j*3 : j*3+3] = rotvec

    return body_pose, global_orient, transl


print("Fonctions de conversion définies.")

## 3. Traitement d'une vidéo complète

In [ ]:
def video_to_smpl(video_path, max_frames=None):
    """
    Extrait les paramètres SMPL approximatifs depuis une vidéo.

    Retourne un dict compatible avec le format hmr4d_results.pt de GVHMR :
        body_pose     : (T, 69)
        global_orient : (T, 3)
        transl        : (T, 3)
        betas         : (10,)  — zéro (forme neutre)
    """
    body_poses, global_orients, transls = [], [], []

    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)

    with mp_pose.Pose(
        model_complexity=1,
        smooth_landmarks=True,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5,
    ) as pose:
        frame_idx = 0
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret or (max_frames and frame_idx >= max_frames):
                break

            rgb     = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = pose.process(rgb)

            if results.pose_world_landmarks:
                lm = results.pose_world_landmarks.landmark
                lm_arr = np.array([[l.x, l.y, l.z, l.visibility] for l in lm])
                bp, go, tr = landmarks_to_smpl_pose(lm_arr)
            else:
                bp = np.zeros(69)
                go = np.zeros(3)
                tr = np.zeros(3)

            body_poses.append(bp)
            global_orients.append(go)
            transls.append(tr)
            frame_idx += 1

            if frame_idx % 100 == 0:
                print(f"  Frame {frame_idx}…", end="\r")

    cap.release()
    print(f"\n{frame_idx} frames traitées")

    smpl_params = {
        "body_pose":     torch.tensor(np.array(body_poses),    dtype=torch.float32),  # (T, 69)
        "global_orient": torch.tensor(np.array(global_orients), dtype=torch.float32), # (T, 3)
        "transl":        torch.tensor(np.array(transls),       dtype=torch.float32),  # (T, 3)
        "betas":         torch.zeros(10, dtype=torch.float32),                        # forme neutre
    }
    return smpl_params, fps


# Test sur la première vidéo disponible
VIDEO_DIR  = Path("videos")
video_list = sorted(VIDEO_DIR.glob("*.mp4"))

if not video_list:
    print("Aucune vidéo trouvée — lancez d'abord le tutoriel 01.")
else:
    print(f"Traitement de : {video_list[0].name}")
    smpl_params, fps = video_to_smpl(str(video_list[0]), max_frames=200)

    for k, v in smpl_params.items():
        print(f"  {k:15s} : {tuple(v.shape)}")

## 4. Sauvegarder au format GVHMR

Le fichier produit a la même structure que `hmr4d_results.pt` de GVHMR.
Il peut être utilisé directement dans le pipeline de conversion ProtoMotions.

In [ ]:
def save_smpl_params(smpl_params, video_path, output_dir="smpl_outputs"):
    """Sauvegarde les paramètres SMPL au format compatible GVHMR."""
    out_dir  = Path(output_dir) / Path(video_path).stem
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / "hmr4d_results.pt"

    # Même structure que la sortie GVHMR
    data = {"smpl_params_global": smpl_params}
    torch.save(data, str(out_path))
    print(f"Sauvegardé : {out_path}")
    return out_path


# Traiter et sauvegarder toutes les vidéos
all_smpl = {}
for video_path in video_list:
    print(f"\n→ {video_path.name}")
    params, fps = video_to_smpl(str(video_path), max_frames=300)
    out = save_smpl_params(params, video_path)
    all_smpl[video_path.stem] = params

print(f"\n{len(all_smpl)} vidéos converties.")

## 5. Visualisation — angles articulaires dans le temps

In [ ]:
if all_smpl:
    name   = list(all_smpl.keys())[0]
    params = all_smpl[name]
    T      = params["body_pose"].shape[0]
    t_axis = np.arange(T) / fps

    # Extraire la norme de rotation de chaque os (= amplitude du mouvement)
    body_pose_np = params["body_pose"].numpy()  # (T, 69)
    magnitudes   = np.linalg.norm(body_pose_np.reshape(T, 23, 3), axis=-1)  # (T, 23)

    JOINT_NAMES_SHORT = [
        "LHip","RHip","Spine","LKnee","RKnee","Chest",
        "LAnkle","RAnkle","Lower","LToe","RToe","Neck",
        "Head","LThorax","RThorax","Nose","LShoulder","RShoulder",
        "LElbow","RElbow","LWrist","RWrist","LHand",
    ]

    # Afficher les 6 os avec le plus de mouvement
    most_active = np.argsort(magnitudes.mean(axis=0))[-6:][::-1]

    fig, axes = plt.subplots(3, 2, figsize=(14, 8), sharex=True)
    for ax, j in zip(axes.flat, most_active):
        ax.plot(t_axis, magnitudes[:, j], lw=1.5)
        ax.set_title(f"{JOINT_NAMES_SHORT[j]} (joint {j})")
        ax.set_ylabel("|rotation| (rad)")
        ax.grid(alpha=0.3)
    axes[-1, 0].set_xlabel("Temps (s)")
    axes[-1, 1].set_xlabel("Temps (s)")
    fig.suptitle(f"Amplitude de rotation — {name[:50]}", fontsize=12)
    plt.tight_layout()
    plt.show()

## Comparaison avec GVHMR

| | Cette approche | GVHMR |
|--|---|---|
| **Méthode** | Mapping géométrique direct | Réseau de neurones + optimisation |
| **Betas (forme)** | Toujours 0 | Estimés par frame |
| **Précision rotations** | Approximative | Précise |
| **Cohérence temporelle** | Bruit frame à frame | Lissée par le modèle |
| **Vitesse** | Temps réel (~30fps) | ~2fps sur GPU |
| **GPU nécessaire** | Non | Oui (8+ GB) |
| **Installation** | `pip install mediapipe` | Env conda complexe |

> **Pour la classification** (tutoriel 4) : cette approximation est **suffisante**.
> Les différences entre gestes (ex. coup droit vs revers) sont assez grandes
> pour être détectées malgré le bruit de la conversion.